In [1]:
# Load and inspect the dataset, then apply binning, summarize, and propose a balanced 30-image subset.
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Paths
csv_path = Path("/Users/foramkamdar/Desktop/oehrn_lab/emotionalPDandRBD/cognitive_reappraisal_task/fk_version_NAPs_list_simplified.csv")

# Load
df = pd.read_csv(csv_path)

# Peek at columns to infer names for IDs, valence, arousal, category
df_columns = df.columns.tolist()

# Display column names for inspection
print("Column names in the dataset:")
for col in df_columns:
    print(col)

Column names in the dataset:
ID
Category
ValenceM
ValenceSD
ArousalM
ArousalSD


In [4]:


# Peek at columns to infer names for IDs, valence, arousal, category
df_columns = df.columns.tolist()

# Try to guess likely column names for key fields
def first_matching(colnames, candidates):
    for c in candidates:
        for col in colnames:
            if col.strip().lower() == c.strip().lower():
                return col
        # also try substring contains
        for col in colnames:
            if c.strip().lower() in col.strip().lower():
                return col
    return None

id_col = first_matching(df_columns, ["Image", "Filename", "File", "ID", "Stimulus", "ImageID", "Name"])
val_col = first_matching(df_columns, ["Valence_M", "Valence", "Valence Mean", "Valence_Mean", "valence_m", "valence"])
aro_col = first_matching(df_columns, ["Arousal_M", "Arousal", "Arousal Mean", "Arousal_Mean", "arousal_m", "arousal"])
cat_col = first_matching(df_columns, ["Category", "category", "Cat", "StimulusCategory", "Class", "Set", "Subset"])

# Ensure numeric conversion for ratings (coerce errors to NaN)
if val_col is not None:
    df[val_col] = pd.to_numeric(df[val_col], errors='coerce')
if aro_col is not None:
    df[aro_col] = pd.to_numeric(df[aro_col], errors='coerce')

# Drop rows missing required ratings
required_cols = [col for col in [val_col, aro_col] if col is not None]
df_clean = df.dropna(subset=required_cols).copy()


In [ ]:

# Apply concept-based bins: 1–2.5 (Low), 2.6–3.5 (Medium), 3.6–5.0 (High).
def bin_scale(x):
    if x <= 2.5:
        return "Low"
    elif 2.6 <= x <= 3.5:
        return "Medium"
    elif x >= 3.6:
        return "High"
    else:
        # Values in gaps (e.g., 2.5<x<2.6 or 3.5<x<3.6) due to decimals—assign to nearest conceptual bin.
        # We'll place 2.51–2.59 into Medium and 3.51–3.59 into High to avoid holes.
        if 2.5 < x < 2.6:
            return "Medium"
        if 3.5 < x < 3.6:
            return "High"
        return np.nan

df_clean["ValenceLevel"] = df_clean[val_col].apply(bin_scale)
df_clean["ArousalLevel"] = df_clean[aro_col].apply(bin_scale)

# Create 3x3 summary table
summary = (df_clean
           .pivot_table(index="ValenceLevel", columns="ArousalLevel", values=val_col, aggfunc='count', fill_value=0)
           .reindex(index=["Low","Medium","High"], columns=["Low","Medium","High"]))

# Now select a balanced subset of 30 images.
# Strategy:
# 1) Target ~ 3 or 4 per cell. We'll start with base target = 3 per cell (total 27), then fill remaining 3 into cells with the fewest samples so far.
# 2) Within each cell, if categories exist, try to spread across categories.
np.random.seed(42)

# Identify an ID-like column for output
if id_col is None:
    # fabricate an ID from index if needed
    id_col = "_row_id"
    df_clean[id_col] = df_clean.index.astype(str)

# Helper: sample from a group with optional category spreading
def balanced_sample_cell(cell_df, n, cat_col):
    if n <= 0 or len(cell_df) == 0:
        return cell_df.iloc[0:0]
    if cat_col is None or cell_df[cat_col].isna().all():
        return cell_df.sample(min(n, len(cell_df)), random_state=42)
    # Try to distribute across categories as evenly as possible
    cats = cell_df[cat_col].fillna("Unknown")
    counts = cats.value_counts().to_dict()
    # order categories by fewer available first to give them a chance
    ordered_cats = sorted(counts.keys(), key=lambda c: counts[c])
    selected_idx = []
    # round-robin pick until we reach n or run out
    cat_iters = 0
    while len(selected_idx) < n and cat_iters < 10*n:
        for c in ordered_cats:
            remaining = cell_df.loc[cell_df.index.difference(selected_idx)]
            pool = remaining[remaining[cat_col].fillna("Unknown") == c]
            if len(pool) > 0 and len(selected_idx) < n:
                # choose 1 from this category
                chosen = pool.sample(1, random_state=42 + len(selected_idx)).index[0]
                selected_idx.append(chosen)
        cat_iters += 1
        if len(selected_idx) >= n or len(cell_df) <= len(selected_idx):
            break
    if len(selected_idx) < n:
        # top up randomly
        remaining = cell_df.loc[cell_df.index.difference(selected_idx)]
        if len(remaining) > 0:
            extra = remaining.sample(min(n - len(selected_idx), len(remaining)), random_state=123).index.tolist()
            selected_idx += extra
    return cell_df.loc[selected_idx]

# First pass: take up to 3 from each cell
targets = {("Low","Low"):3,("Low","Medium"):3,("Low","High"):3,
           ("Medium","Low"):3,("Medium","Medium"):3,("Medium","High"):3,
           ("High","Low"):3,("High","Medium"):3,("High","High"):3}

selected_parts = []
cell_availability = {}

for v in ["Low","Medium","High"]:
    for a in ["Low","Medium","High"]:
        cell = df_clean[(df_clean["ValenceLevel"]==v) & (df_clean["ArousalLevel"]==a)]
        cell_availability[(v,a)] = len(cell)
        take_n = min(targets[(v,a)], len(cell))
        selected_parts.append(balanced_sample_cell(cell, take_n, cat_col))

selected_df = pd.concat(selected_parts, axis=0) if selected_parts else df_clean.iloc[0:0]

# Second pass: add remaining to reach 30, preferring cells we haven't sampled much and that have availability
remain = max(0, 30 - len(selected_df))
if remain > 0:
    # Rank cells by current take/availability (prefer those with more availability left and fewer already taken)
    current_counts = selected_df.groupby(["ValenceLevel","ArousalLevel"]).size()
    current_counts = current_counts.reindex(pd.MultiIndex.from_product([["Low","Medium","High"],["Low","Medium","High"]]), fill_value=0)
    # Build an ordered list of cells to draw extras from: those with lowest current count but with remaining availability
    cells_order = sorted(cell_availability.keys(), key=lambda k: (current_counts[k], -cell_availability[k]))
    for (v,a) in cells_order:
        if remain <= 0:
            break
        cell = df_clean[(df_clean["ValenceLevel"]==v) & (df_clean["ArousalLevel"]==a)]
        already_ids = set(selected_df[id_col].tolist())
        pool = cell[~cell[id_col].isin(already_ids)]
        if len(pool) == 0:
            continue
        # take up to min(2, remain) to prevent one cell dominating
        take_n = min(2, remain, len(pool))
        extra = balanced_sample_cell(pool, take_n, cat_col)
        selected_df = pd.concat([selected_df, extra], axis=0)
        remain = 30 - len(selected_df)

# Final: if still short (e.g., sparse dataset), top-up from global pool not yet selected
if len(selected_df) < 30:
    pool = df_clean[~df_clean[id_col].isin(set(selected_df[id_col].tolist()))]
    if len(pool) > 0:
        topup = pool.sample(min(30 - len(selected_df), len(pool)), random_state=999)
        selected_df = pd.concat([selected_df, topup], axis=0)

# Prepare outputs
labeled_cols = [id_col]
if cat_col is not None: labeled_cols.append(cat_col)
labeled_cols += [val_col, aro_col, "ValenceLevel", "ArousalLevel"]

labeled_output = df_clean[labeled_cols].copy()

# Summary table as DataFrame with totals
summary_df = summary.copy()
summary_df.loc["Total"] = summary_df.sum(axis=0)
summary_df["Total"] = summary_df.sum(axis=1)
grand_total = int(summary_df.loc["Total","Total"]) if "Total" in summary_df.index else int(df_clean.shape[0])

# Save artifacts
labeled_path = Path("/Users/foramkamdar/Desktop/oehrn_lab/emotionalPDandRBD/cognitive_reappraisal_task/stimuli_labeled_bins.csv")
summary_path = Path("/Users/foramkamdar/Desktop/oehrn_lab/emotionalPDandRBD/cognitive_reappraisal_task/stimuli_3x3_summary.csv")
subset_path = Path("/Users/foramkamdar/Desktop/oehrn_lab/emotionalPDandRBD/cognitive_reappraisal_task/stimuli_balanced_subset30.csv")


labeled_output.to_csv(labeled_path, index=False)
summary_df.to_csv(summary_path)
selected_df[labeled_cols].to_csv(subset_path, index=False)


In [ ]:

# Display key results to the user
from caas_jupyter_tools import display_dataframe_to_user

display_dataframe_to_user("Labeled stimuli (first 200 rows)", labeled_output.head(200))
display_dataframe_to_user("3x3 summary table (counts)", summary_df.reset_index())
display_dataframe_to_user("Proposed balanced 30-image subset", selected_df[labeled_cols].reset_index(drop=True))

{
    "columns_detected": {
        "id_col": id_col,
        "valence_col": val_col,
        "arousal_col": aro_col,
        "category_col": cat_col
    },
    "counts": {
        "total_rows": int(df.shape[0]),
        "rows_with_ratings": int(df_clean.shape[0]),
        "subset_size": int(selected_df.shape[0]),
        "grand_total_summary": grand_total
    },
    "output_files": {
        "labeled_csv": str(labeled_path),
        "summary_csv": str(summary_path),
        "subset_csv": str(subset_path)
    }
}
